# (5) Transfer Learning

Surrogate models are useful only if they can be evaluated on new designs without re-running
FEMM. Two practical wishlist items go beyond the η-map predictor from Chapter 4:

1. **New target on the same designs.** Engineers care about power factor too — but the η
   `MapDecoder` only outputs η. Can we fine-tune it for PF using less data than training
   from scratch?
2. **Same target on a new motor topology.** What if we want maps for a *different machine*
   (e.g. a 12-slot/10-pole surface-mount FSCW PM)? Can knowledge from 2511 IPM designs
   bootstrap the new model with far fewer FSCW designs?

Both are classic **transfer learning** problems. This chapter answers them concretely with
the IPM η-`MapDecoder` weights from Chapter 4 as the pretrained source.


## 5.1  Why transfer learning matters for surrogate models

Every FEMM solve costs 30–60 seconds; a per-design pipeline (Phases 1–4) takes 1–2 minutes.
2511 IPM designs took roughly two days of wall-time on Windows. For a new task — different
output or different motor — training a fresh model from scratch would naively require
*another* full FEMM sweep.

If a pretrained model transfers, two things change:

- The pretrained network already encodes *generic* design→(spatial-pattern) features in its
  hidden layers; the new task only needs to relearn the output-specific mapping.
- The data requirement collapses, *at least at small n*. With careful fine-tuning, a few
  dozen new designs can land at an accuracy where from-scratch needs several times more.

The danger: **negative transfer**. If the pretrained output distribution is far from the new
target's distribution, the model can land in a worse local minimum than random init. We'll
see exactly this happen in §5.4 and how to fix it.


## 5.2  Task transfer: η → power factor

Same 2511 IPM designs, same operating-point grid, different output. The power factor map is
derivable from the existing Phase 3 saved data — no new FEMM runs:

$$
u_d = R_s i_d - \omega_e \lambda_q, \qquad
u_q = R_s i_q + \omega_e \lambda_d, \qquad
|\mathrm{PF}| = \frac{|u_d i_d + u_q i_q|}{|\mathbf{u}|\,|\mathbf{i}|}
$$

All four ingredients ($i_d$, $i_q$, $\lambda_d$, $\lambda_q$) are in `phase3_eta_points.csv`,
and ($R_s$, pole pairs) come from the shared `Config`.

**Convention note.** The dq voltage equation above produces a signed real-power expression
with the *opposite* sign from textbook motoring convention in this codebase — a side effect
of the Park q-axis sign fix recorded in `plan_May22.md §6`. We report the magnitude
$|\mathrm{PF}|$ (the *displacement* power factor used industrially) rather than the signed
value. The TL story holds either way — we just need a deterministic target distinct from η.
The 2511-design PF master dataset has mean $|\mathrm{PF}| \approx 0.12$ and range
$[0, 0.95]$ inside the operating envelope.


Run once on the existing 2511-design dataset to write the PF master NPZ:

```powershell
python -m data_gen.compute_pf_maps `
    --dataset-root data_gen/outputs/dataset_train
```

Takes about 4 minutes for 2511 designs. Produces `master_pf_dataset.npz` with the same
schema as `master_dataset.npz` except `eta_norm_grids` is replaced by `pf_norm_grids`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path('data_gen/outputs')
eta_master = np.load(ROOT / 'dataset_train' / 'master_dataset.npz', allow_pickle=False)
pf_master  = np.load(ROOT / 'dataset_train' / 'master_pf_dataset.npz', allow_pickle=False)

eta = eta_master['eta_norm_grids']
pf  = pf_master['pf_norm_grids']
print(f'eta shape: {eta.shape}, range inside envelope: '
      f'[{np.nanmin(eta):.3f}, {np.nanmax(eta):.3f}], mean={np.nanmean(eta):.3f}')
print(f'pf  shape: {pf.shape},  range inside envelope: '
      f'[{np.nanmin(pf):.3f}, {np.nanmax(pf):.3f}], mean={np.nanmean(pf):.3f}')


## 5.3  Visual comparison: η-map vs PF-map for matched designs

Pick four designs across the $T_\text{max}$ spread; show the η map and the PF map
side by side. They look different — the PF map is dominated by low values at
low speed (where $\omega_e$ is small and the resistive voltage drop dominates) and
peaks at high-speed light load. The η map peaks in the mid-band. **Different
patterns → genuinely different targets**.


In [ ]:
N_axis = eta_master['N_norm_grid']
T_axis = eta_master['T_norm_grid']
T_max  = eta_master['T_max_Nm']

order = np.argsort(T_max)
picks = [order[0], order[len(order)//3], order[2*len(order)//3], order[-1]]

fig, axes = plt.subplots(len(picks), 2, figsize=(11, 4*len(picks)))
for row, idx in enumerate(picks):
    for col, (img, label, cmap, vmax) in enumerate([
        (eta[idx], 'eta', 'viridis', 1.0),
        (pf[idx],  '|PF|', 'plasma',  1.0),
    ]):
        ax = axes[row, col]
        im = ax.imshow(img, origin='lower', aspect='auto',
                       extent=[N_axis[0], N_axis[-1], T_axis[0], T_axis[-1]],
                       cmap=cmap, vmin=0, vmax=vmax)
        ax.set_xlabel('$N / N_\\mathrm{max}$')
        ax.set_ylabel('$T / T_\\mathrm{max}$')
        ax.set_title(f'{label}  ($T_\\mathrm{{max}}$={T_max[idx]:.1f} Nm)')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()


## 5.4  Sample-efficiency curve

Three training modes:

- **scratch** — random `MapDecoder` init, train on the PF master.
- **pretrained-naive** — load the IPM η `MapDecoder` fold-1 weights, fine-tune all
  parameters.
- **pretrained-reset** — load the IPM η weights, but *re-initialise* the final
  ConvTranspose layer (the output head) before fine-tuning. Keeps the feature
  extractor, lets the output mapping re-learn from scratch.

Each mode is trained at $n_\text{train} \in \{10, 25, 50, 100, 200, 400\}$ with 3 seeds
per setting. Held-out designs from the 2511-design PF master become the validation set.
Total: 54 runs at ~100 s each (~90 min on CPU).

```powershell
python -m data_gen.ml.run_phase5_pf_sweep `
    --dataset data_gen/outputs/dataset_train/master_pf_dataset.npz `
    --out-dir data_gen/outputs/phase5_pf_train `
    --pretrained data_gen/outputs/phase5_map_train/weights_fold1.pt `
    --epochs 1500
```

Outputs `data_gen/outputs/phase5_pf_train/sweep_metrics.csv` *and* per-run
`metrics.csv` files. The next cell collects whichever runs are available on disk —
you can re-execute it after the sweep finishes or rerun the sweep with different
knobs and the plot will update.


In [ ]:
# Gather all completed per-run metrics — robust to a partial sweep.
# This points at the width-4x pretrained-source sweep (see §4.7.3 / §7.7 for the
# architecture lift). The baseline-source sweep still lives at
# `phase5_pf_train/` for comparison; switching the path below reproduces the
# original story.
PF_DIR = ROOT / 'phase5_pf_train_width4x'

metrics_files = sorted(PF_DIR.glob('n*_s*_*/metrics.csv'))
if not metrics_files:
    raise FileNotFoundError(
        f'No per-run metrics.csv found under {PF_DIR}. '
        'Run `python -m data_gen.ml.run_phase5_pf_sweep --pretrained '
        'data_gen/outputs/phase5_map_train_arch/width4x/weights_fold1.pt '
        '--arch-preset width4x --out-dir data_gen/outputs/phase5_pf_train_width4x` first.'
    )
sweep = pd.concat([pd.read_csv(p) for p in metrics_files], ignore_index=True)
# Coerce types (sometimes 'mode' is auto-detected as bool for some Pandas versions).
sweep['mode'] = sweep['mode'].astype(str)
sweep['val_r2'] = sweep['val_r2'].astype(float)
sweep['n_train'] = sweep['n_train'].astype(int)
sweep['seed'] = sweep['seed'].astype(int)

# Drop any duplicate (n_train, seed, mode) rows in case the sweep was run more than once.
sweep = sweep.drop_duplicates(subset=['n_train', 'seed', 'mode'], keep='last')

n_runs = len(sweep)
modes_present = sorted(sweep['mode'].unique())
n_sizes = sweep['n_train'].nunique()
n_seeds = sweep['seed'].nunique()
expected = n_sizes * n_seeds * len(modes_present)
print(f'Loaded {n_runs} runs from {len(metrics_files)} metrics.csv files')
print(f'Pretrained source: {PF_DIR.name}  (width-4x MapDecoder, R^2 = 0.973 on IPM eta)')
print(f'Coverage: {n_sizes} sizes x {n_seeds} seeds x {len(modes_present)} modes '
      f'= {expected} expected; {n_runs} present ({n_runs/expected:.0%})')
print(f'Modes: {modes_present}')
print(f'Sizes: {sorted(sweep["n_train"].unique())}')
if n_runs < expected:
    print(f'\n[NOTE] Partial sweep — the figure below will use only what is available.')

In [ ]:
# Sample-efficiency curve from whatever data is present.
PREFERRED_ORDER = ['scratch', 'pretrained-naive', 'pretrained-reset']
modes = [m for m in PREFERRED_ORDER if m in modes_present]
modes += [m for m in modes_present if m not in PREFERRED_ORDER]
colors = {'scratch': 'C0', 'pretrained-naive': 'C1', 'pretrained-reset': 'C2'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
for mode in modes:
    sub = sweep[sweep['mode'] == mode]
    if sub.empty:
        continue
    agg = sub.groupby('n_train')['val_r2'].agg(['mean', 'std']).reset_index()
    ax.errorbar(agg['n_train'], agg['mean'],
                yerr=agg['std'].fillna(0),
                marker='o', lw=2, capsize=4, label=mode,
                color=colors.get(mode))
ax.set_xscale('log')
ax.set_xlabel('Training designs')
ax.set_ylabel('Held-out R²')
ax.set_title(f'Sample-efficiency curve  ({n_seeds} seed{"s" if n_seeds!=1 else ""}, error bars = ±1σ)')
ax.axhline(0, color='gray', lw=0.5)
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
small_n = sweep['n_train'].min()
small = sweep[sweep['n_train'] == small_n]
positions = np.arange(len(modes))
means = [small[small['mode'] == m]['val_r2'].mean() for m in modes]
stds  = [small[small['mode'] == m]['val_r2'].std()  for m in modes]
ax.bar(positions, means, yerr=stds, capsize=6,
       color=[colors.get(m, 'gray') for m in modes], alpha=0.85)
ax.set_xticks(positions); ax.set_xticklabels(modes, rotation=15)
ax.set_ylabel('Held-out R²')
ax.set_title(f'Head-to-head at smallest available n_train = {small_n}')
ax.axhline(0, color='gray', lw=0.5); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

print('\nMean held-out R² per (mode, n_train):')
tbl = sweep.groupby(['n_train', 'mode'])['val_r2'].agg(['mean', 'std', 'count']).round(3)
print(tbl.to_string())


### How to read the curve

The 2511-design retrain with the **width-4× pretrained source** (633 k params,
$R^2=0.973$ on IPM η, see §4.7.3) produced this aggregate picture (mean ± std
over 3 seeds):

| $n_\text{train}$ | scratch | pretrained-naive | pretrained-reset | reset − scratch |
|---:|:---:|:---:|:---:|:---:|
| 10  | $-0.32 \pm 1.27$ | $+0.63 \pm 0.06$ | $+0.31 \pm 0.25$ | **+0.63** |
| 25  | $+0.63 \pm 0.09$ | $+0.76 \pm 0.01$ | $+0.71 \pm 0.03$ | **+0.08** |
| 50  | $+0.78 \pm 0.01$ | $+0.79 \pm 0.00$ | $+0.82 \pm 0.01$ | +0.04 |
| 100 | $+0.84 \pm 0.01$ | $+0.79 \pm 0.00$ | $+0.86 \pm 0.02$ | +0.02 |
| 200 | $+0.87 \pm 0.01$ | $+0.80 \pm 0.01$ | $+0.88 \pm 0.01$ | +0.01 |
| 400 | $+0.88 \pm 0.01$ | $+0.81 \pm 0.01$ | $+0.88 \pm 0.01$ | +0.01 |

Three things to take from this:

1. **Naive transfer is no longer broken.** With the baseline-pretrained source
   the naive path was pinned at $R^2 \approx 0.07$ across every $n$ — the
   pretrained-η output bias dragged it into a degenerate local minimum that
   gradient descent could not escape. With the *width-4×* pretrained source the
   naive path now climbs to $R^2 \approx 0.81$ at $n=400$. The wider feature
   extractor carries enough task-agnostic structure that fine-tuning can
   over-write the output bias even without an explicit head reset. This is
   the §7.8 "rerunning Chapter 5 with width-4× weights may shift the
   cross-domain story" result — and the shift is large.
2. **Reset still beats naive at every $n$**, but the gap is now small (0.88 vs
   0.81 at $n=400$ — a $-0.07$ penalty for not resetting, where with the
   baseline source it was a $-0.8$ penalty). The output-distribution mismatch
   (η ≈ 0.7 → PF ≈ 0.12) still costs accuracy, just less catastrophically.
3. **Small-$n$ benefit is now huge.** `pretrained-reset` beats scratch by
   **+0.63 at $n=10$** (scratch crashes to $R^2=-0.32$ — it cannot find any
   useful predictor from 10 designs alone, while reset's pretrained feature
   extractor is enough to push past $R^2=+0.3$). At $n=25$ the reset advantage
   is +0.08; by $n=100$ it shrinks to +0.02; by $n=400$ scratch and reset are
   tied at $R^2=0.88$.

**Practical takeaway.** With the upgraded pretrained source, the TL benefit
window widens from "$n \lesssim 50$" (baseline source) to "$n \lesssim 100$"
(width-4× source). Reset remains the safe production recipe — even at $n=400$
it never underperforms scratch — and the "always reset the output head"
heuristic now has a more nuanced backing: it's not about avoiding catastrophe
(naive isn't broken any more) but about extracting the last percentage point
from the cross-task transfer.

## 5.5  Per-design comparison at fixed budget

At $n_\text{train}=50$ (a realistic small-budget scenario), which designs see the
biggest TL benefit? Load predictions from one matched scratch and pretrained-reset
run and plot per-design RMSE. Skips automatically if either run is missing.


In [ ]:
sc_path = PF_DIR / 'n0050_s0_scratch' / 'predictions.npz'
pt_path = PF_DIR / 'n0050_s0_pretrained-reset' / 'predictions.npz'

if not (sc_path.exists() and pt_path.exists()):
    print(f'Skipping per-design comparison — need both:\n  {sc_path}\n  {pt_path}')
    print('Run the n_train=50 / seed=0 entries of the sweep to populate them.')
else:
    sc = np.load(sc_path, allow_pickle=False)
    pt = np.load(pt_path, allow_pickle=False)

    def per_design_rmse(pred, true, mask):
        out = []
        for i in range(pred.shape[0]):
            m = mask[i]
            if m.sum() == 0: out.append(np.nan); continue
            d = (pred[i] - true[i])[m]
            out.append(float(np.sqrt(np.mean(d**2))))
        return np.array(out)

    rmse_sc = per_design_rmse(sc['val_pred'], sc['val_true'], sc['val_mask'])
    rmse_pt = per_design_rmse(pt['val_pred'], pt['val_true'], pt['val_mask'])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    ax = axes[0]
    lim = max(np.nanmax(rmse_sc), np.nanmax(rmse_pt)) * 1.05
    ax.scatter(rmse_sc, rmse_pt, s=8, alpha=0.4)
    ax.plot([0, lim], [0, lim], 'k--', lw=1, label='equal')
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel('scratch per-design RMSE')
    ax.set_ylabel('pretrained-reset per-design RMSE')
    ax.set_title('Per-design error (n_train=50). Below line = TL helps')
    below = int((rmse_pt < rmse_sc).sum()); above = int((rmse_pt > rmse_sc).sum())
    ax.text(0.02, 0.98, f'TL better: {below}\nscratch better: {above}',
            transform=ax.transAxes, va='top', fontsize=11,
            bbox=dict(facecolor='white', alpha=0.8))
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1]
    delta = rmse_sc - rmse_pt  # positive = TL helps
    ax.hist(delta, bins=40, edgecolor='black', alpha=0.7)
    ax.axvline(0, color='r', lw=1)
    ax.set_xlabel('Δ RMSE  =  scratch − pretrained-reset')
    ax.set_ylabel('Designs')
    ax.set_title(f'mean Δ = {np.nanmean(delta):+.4f}, median = {np.nanmedian(delta):+.4f}')
    ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()


## 5.6  Layer-freezing study  *(optional follow-up)*

`pretrained-reset` keeps the entire backbone trainable. Two extremes worth checking:

- **Freeze `head`** (the two Linear layers): only the decoder learns.
- **Freeze `decoder`** (the two ConvTranspose layers): only the head learns.

Both are achievable with the existing CLI:

```powershell
python -m data_gen.ml.run_phase5_pf `
    --n-train 50 --seed 0 `
    --pretrained data_gen/outputs/phase5_map_train/weights_fold1.pt `
    --reset-output `
    --freeze head
```

Run the two freezing variants at the same `(n_train, seed)` as the
`pretrained-reset` run above, then re-execute the cell in §5.4 — the new metrics
will be picked up automatically by the glob and shown as additional bars/lines.

> **TODO when results are in**: discuss which freeze setting wins and what that
> says about where the IPM-pretrained features actually live in the network.


## 5.7  Domain transfer: FSCW PM motor (12-slot / 10-pole, surface-magnet)

The IPM port covers one machine — a 12-slot/4-pole distributed-winding interior-PM
motor. The paper's transfer-learning chapter compares against a *second* machine:
a 12-slot/10-pole **fractional-slot concentrated-winding** (FSCW) PM motor with
surface-mount magnets. Different topology, different pole count, different winding
pattern — but the surrogate has the same input/output shape (6 design params
→ 60×80 normalised map). That shape-match is exactly the condition the
`MapDecoder` checkpoint needs to be reusable across topologies.

The open-source FSCW pipeline is wired in `data_gen/`:

| File | What it does |
|---|---|
| `geometry_fscw.py` | Builds the 12s/10p surface-PM motor in FEMM (`build_fscw`). |
| `config.py:FSCWGeometry`, `FSCWWindingPattern` | Surface-magnet geometry; standard 12s/10p AA'CC'BB' double-layer winding (winding factor ~0.933). |
| `design_space.py:lhs_designs_fscw` | LHS sampler over the FSCW design vector: $\{R_\text{stator,outer}, R_\text{stator,bore}, \text{slot depth}, \text{magnet thickness}, \text{magnet width}, \text{stack length}\}$. |
| `dataset_runner.py`, `run_dataset.py --motor fscw` | Same orchestration as IPM, branching on `motor_type`. |

::::{admonition} FSCW dataset ready (2026-05-27)
:class: tip

The FEMM/COM crash that blocked the original 500-design sweep has been resolved
(geometry-side fix). The current FSCW master at
`outputs/dataset_fscw/master_dataset.npz` contains **49 valid designs** out of
~46 attempted LHS samples — small but enough to exercise the cross-topology
transfer-learning question below.
::::

Schema parity with the IPM master is what makes the cross-topology transfer mechanical:
the only renamed parameter is `magnet_length` → `magnet_thickness` (same physical knob,
different convention for radial- vs axially-extruded magnets), and the η-map grid
shape and normalisation scheme are identical.

## 5.8  IPM → FSCW domain-transfer experiment

Same sweep design as §5.4, with three differences from the IPM→PF story:

1. **Source = IPM η**, target = FSCW η. The output *distribution* is preserved
   (both are normalised efficiency in [0, 1] with similar means in the 0.7–0.95
   range inside the envelope), only the *machine topology* changes.
2. **Tiny target dataset**: 49 designs total, so the size sweep runs at
   $n_\text{train} \in \{5, 10, 15, 20, 25, 30, 35\}$ with the remainder as
   the held-out validation set. Each run is a single train/val split, not a
   k-fold — 3 random seeds per cell to size the noise floor.
3. **Pretrained source**: `phase5_map_train/weights_fold1.pt` (IPM η MapDecoder
   trained on 2511 designs).

The prediction: where the §5.4 PF transfer was *killed* by the output-distribution
mismatch (η~0.7 → PF~0.12), this transfer should *succeed* without resetting the
output layer, because the new target lives in the same value range as the
pretraining target. If true, `pretrained-naive` should beat `pretrained-reset` at
every $n$ — opposite ordering to §5.4.

**Anchor: standalone FSCW MapDecoder.** Independent of any transfer story, a
5-fold CV on the same 49 FSCW designs at the baseline (157 k) `MapDecoder`
topology hits **OOF $R^2 = 0.835$** (RMSE = 0.081, MAE = 0.045), output at
`outputs/phase5_map_fscw_train/cv_metrics.csv`. This is the from-scratch
baseline the TL sweep below has to beat to claim transfer matters; the IPM
counterpart at the same architecture and a much larger 2511-design dataset hits
$R^2 = 0.913$ (§4.7.3) — FSCW with $\sim 50\times$ less data lands only
$\Delta R^2 \approx -0.08$ behind. The architecture is more topology-agnostic
than the data-quantity asymmetry suggests.

:::{admonition} Future work — rerun §5.8 with the width-4× pretrained source
:class: tip
**Expected wall:** ~6 hours on CPU (63 runs × ~5.5 min/run, scaling from the width-4× PF sweep wall in `phase5_pf_train_width4x/sweep_metrics.csv`).
**Expected advantage:** would test whether the width-4× $R^2$ lift (0.913 → 0.973 on IPM η) survives the cross-topology jump to FSCW. Could push pretrained-naive R² beyond the current ~0.84 ceiling at $n=35$, or expose that width-4×'s extra capacity overfits at FSCW's 49-design scale. Either result is a real finding — does cross-task transfer (IPM→PF) and cross-topology transfer (IPM→FSCW) benefit *equally* from the upgraded source?
**Command:**
```bash
python -m data_gen.ml.run_phase5_fscw_sweep \
  --pretrained data_gen/outputs/phase5_map_train_arch/width4x/weights_fold1.pt \
  --arch-preset width4x \
  --out-dir data_gen/outputs/phase5_fscw_train_width4x
```
:::

:::{admonition} Future work — standalone FSCW 5-fold CV at width-4×
:class: tip
**Expected wall:** ~25 minutes on CPU (5 folds × ~5 min each at width-4× on 49 designs).
**Expected advantage:** anchors the §5.8 TL story against a from-scratch baseline at *both* architecture scales. Standalone width-4× FSCW R² > standalone baseline FSCW R² would confirm the §4.7.3 architectural lift is topology-agnostic — the depth+1 / width-2× / width-4× pattern observed on 2511 IPM designs reappears on 49 FSCW designs. Standalone CV at baseline is already done (`phase5_map_fscw_train/cv_metrics.csv` → OOF $R^2 = 0.835$); the width-4× counterpart closes the comparison.
**Command:**
```bash
python -m data_gen.ml.run_phase5_map \
  --dataset data_gen/outputs/dataset_fscw/master_dataset.npz \
  --out-dir data_gen/outputs/phase5_map_fscw_train_width4x \
  --arch-preset width4x --folds 5 --epochs 1500 --weight-decay 1e-4
```
:::

In [ ]:
# IPM → FSCW domain-transfer sweep: aggregate metrics and sample-efficiency curve.
# Reads sweep_metrics.csv written by `python -m data_gen.ml.run_phase5_fscw_sweep`.

FSCW_DIR = ROOT / 'phase5_fscw_train'
fscw_master = ROOT / 'dataset_fscw' / 'master_dataset.npz'

if not fscw_master.exists():
    print('[skip] FSCW master dataset not found at:', fscw_master)
elif not FSCW_DIR.exists():
    z = np.load(fscw_master, allow_pickle=False)
    print(f'FSCW master loaded: {z["design_params"].shape[0]} designs, '
          f'params={list(z["param_names"])}')
    print('[skip] Sweep not run yet. Launch with:')
    print('  python -m data_gen.ml.run_phase5_fscw_sweep \\')
    print('      --n-train-sizes 5 10 15 20 25 30 35 --seeds 0 1 2')
else:
    metrics_files = sorted(FSCW_DIR.glob('n*_s*_*/metrics.csv'))
    if not metrics_files:
        print(f'[skip] No per-run metrics found under {FSCW_DIR}.')
    else:
        fscw_sweep = pd.concat([pd.read_csv(p) for p in metrics_files], ignore_index=True)
        print(f'Loaded {len(fscw_sweep)} FSCW transfer runs from {FSCW_DIR.name}/.')

        # Aggregate
        agg = (fscw_sweep
               .groupby(['n_train', 'mode'])['val_r2']
               .agg(['mean', 'std', 'count'])
               .round(3))
        print()
        print('val_r2 by (n_train, mode):')
        print(agg.to_string())

        # Pivot to wide form for the plot
        means = (fscw_sweep
                 .pivot_table(index='n_train', columns='mode', values='val_r2', aggfunc='mean')
                 .reindex(columns=['scratch', 'pretrained-naive', 'pretrained-reset']))
        stds = (fscw_sweep
                .pivot_table(index='n_train', columns='mode', values='val_r2', aggfunc='std')
                .reindex(columns=means.columns))

        fig, ax = plt.subplots(figsize=(7.2, 4.6))
        colors = {'scratch': 'tab:gray',
                  'pretrained-naive': 'tab:blue',
                  'pretrained-reset': 'tab:orange'}
        for col in means.columns:
            if col not in fscw_sweep['mode'].unique():
                continue
            m = means[col].values
            s = stds[col].fillna(0.0).values
            ax.plot(means.index, m, marker='o', color=colors[col], label=col)
            ax.fill_between(means.index, m - s, m + s, color=colors[col], alpha=0.15)
        ax.set_xlabel(r'$n_\mathrm{train}$ (FSCW designs)')
        ax.set_ylabel(r'val masked $R^2$ on FSCW η-map')
        ax.set_title('IPM → FSCW domain transfer  (3 seeds, 1500 epochs)')
        ax.grid(True, alpha=0.3)
        ax.legend(loc='lower right')
        ax.set_ylim(0.0, 1.0)
        plt.tight_layout()
        plt.show()

## 5.9  Limitations and what's next

- **The output distribution decides whether `naive` transfer works.** In the
  IPM→PF task transfer the pretrained η output bias actively hurts PF prediction
  — `pretrained-naive` is pinned at $R^2 \approx 0.07$ from $n=25$ upward, an
  unusable predictor — but in the IPM→FSCW *domain* transfer in §5.8 the source
  and target outputs are both η-maps, and `pretrained-naive` wins at every $n$.
  The practical rule: reset the output head iff the pretraining target lives in
  a different value range than the new target.
- **TL most useful when target data is scarcest.** In both experiments the
  pretrained-mode advantage is biggest at the smallest $n_\text{train}$ and
  narrows as data accumulates. For IPM→PF, scratch ties pretrained-reset by
  $n=100$ (out of 2511 designs available). For IPM→FSCW, pretrained-naive still
  leads scratch by ~$+0.04$ at $n=35$ — but FSCW only has 49 designs total, so
  the curves have not yet shown a crossover, and the gap *narrows* monotonically
  from $+0.05$ at $n=10$ to $+0.04$ at $n=35$.
- **Capacity-bound ceiling, again.** The IPM→PF curves plateau at
  $R^2 \approx 0.83$ at $n=400$; the IPM→FSCW curves plateau at $\approx 0.84$
  at $n=35$. Same architectural ceiling visible in §4.7.1 and §7.7 — the
  baseline `MapDecoder` (157 k params) is the bottleneck, not the data. Wider/
  deeper decoder variants (`width2x`, `width4x`, `depth1` in
  `MAPDECODER_PRESETS`) are the natural next sweep.
- **Single pretrain source.** We use only fold-1 of the η `MapDecoder` as the
  pretrain. A small improvement: average the 5 fold checkpoints into a single
  ensemble init, or run the TL sweep with each fold as a different pretrain
  source. We did not since the gap was already clear.
- **FSCW dataset is tiny.** 49 designs is enough to read off the TL ordering
  cleanly but not to nail down whether scratch eventually catches pretrained-naive.
  Re-running the §5.8 sweep with a few-hundred-design FSCW master would settle
  whether the cross-topology gap closes the way the cross-task gap did.
- **The PF "convention quirk".** $|\mathrm{PF}|$ as defined here is a
  deterministic function of operating point and design but is *not* the physical
  displacement factor in the textbook sense (the dq voltage formula in this
  codebase has a sign flip relative to the standard motoring convention). The TL
  story is unaffected; the absolute value of PF readings should not be quoted as
  a physical PF in downstream comparisons.

Beyond this chapter, two natural extensions are *multi-task* learning (predict η
and PF jointly from a shared backbone) and *adversarial domain adaptation* (DANN)
to soften the topology gap between IPM and FSCW. Both reuse the infrastructure
built here.